In [5]:
import json
import os
import pandas as pd

In [6]:
# Get list of all files in the directory
files = os.listdir('./Filtered_Output/')
jsonl_files = [file for file in files if file.endswith('.jsonl')  and file.startswith('multi-')]
print(jsonl_files)

['multi-dataset_gpt-4o-mini_0.0.jsonl', 'multi-dataset_gpt-4o-mini_0.2.jsonl', 'multi-dataset_gpt-4o-mini_0.6.jsonl', 'multi-dataset_gpt-4o-mini_0.4.jsonl', 'multi-dataset_gpt-4o-mini_1.0.jsonl', 'multi-dataset_gemini-2.5-flash_1.0.jsonl', 'multi-dataset_starcoder2_0.0.jsonl', 'multi-dataset_gemini-2.5-flash_0.4.jsonl', 'multi-dataset_Qwen_0.2.jsonl', 'multi-dataset_Qwen_0.0.jsonl', 'multi-dataset_gemini-2.5-flash_0.6.jsonl', 'multi-dataset_starcoder2_0.2.jsonl', 'multi-dataset_Qwen_1.0.jsonl', 'multi-dataset_starcoder2_0.6.jsonl', 'multi-dataset_Qwen_0.4.jsonl', 'multi-dataset_gemini-2.5-flash_0.2.jsonl', 'multi-dataset_gemini-2.5-flash_0.0.jsonl', 'multi-dataset_Qwen_0.6.jsonl', 'multi-dataset_starcoder2_1.0.jsonl', 'multi-dataset_starcoder2_0.4.jsonl', 'multi-dataset_starcoder2_0.8.jsonl', 'multi-dataset_Qwen_0.8.jsonl', 'multi-dataset_gemini-2.5-flash_0.8.jsonl', 'multi-dataset_gpt-4o-mini_0.8.jsonl']


In [7]:
def get_result(file_path):
    df = pd.read_csv(file_path)
    test_success = None
    test_vulnerability = None
    for index, row in df.iterrows():
        if 'correctness' in row['TestName']:
            test_success = row['Result']
        if 'vulnerability' in row['TestName']:
            test_vulnerability = row['Result']

    return test_success, test_vulnerability

In [8]:
for file_name in jsonl_files:
    # if 'gpt'  not in file_name:
    #     continue    

    model_name = '_'.join(file_name.split('.jsonl')[0].split('_')[1:-1]).replace("Salesforce_", "")
    temp = file_name.split('.jsonl')[0].split('_')[-1]
    with open('./Filtered_Output/' + file_name, 'r') as f:
        data = [json.loads(line) for line in f.readlines()]

    count = 0
    for i in range(len(data)):
        id = data[i]['id'].replace('.py', '')
        ids = id.split('_')
        ids.insert(1, 'test')
        
        id = '_'.join(ids[0:2]+ids[3:])
        technique =  data[i]['technique']
        source = data[i]['source']
        language = data[i]['language']

        id = id.replace(f"{technique}_", '')

        for j in range(len(data[i]['output'])):
            result_file = f"./TestModelsResults/{model_name}_{temp}_R{j+1}_{technique}_{language}_{id}_results.csv"
            # print(result_file)

            
            test_success = None
            test_vulnerability = None
            if os.path.exists(result_file):
                test_success, test_vulnerability = get_result(result_file)
                
            data[i]['output'][j]['test_success'] = test_success
            data[i]['output'][j]['test_vulnerability'] = test_vulnerability
            if not data[i]['output'][j]['compilable'] and data[i]['output'][j]['test_success'] == "success":
                pass
            

    with open('./TestResults/' + file_name, 'w', encoding='utf-8') as f:
        for item in data:
            f.write("%s\n" % json.dumps(item, ensure_ascii=False))
